# Ch 21. RLHF — Solutions

> This notebook provides reproducible solutions to all five exercises. Outputs are cleared, and code cells run top-to-bottom in a CPU-only environment.


## Problem 1

**Problem**: Train a toy environment such as CartPole with REINFORCE.

### Solution

To avoid an external environment dependency, use a two-action bandit as the toy policy. REINFORCE minimizes $-(R-b)\log\pi(a)$; the baseline reduces variance without changing the expectation. The seeded run confirms that probability of the better action increases.


In [ ]:
import numpy as np

rng=np.random.default_rng(2101); logits=np.zeros(2); rewards=np.array([0.2,0.8]); history=[]
for step in range(800):
    probs=np.exp(logits-logits.max()); probs/=probs.sum(); action=rng.choice(2,p=probs)
    reward=rewards[action]+rng.normal(scale=.05); advantage=reward-probs@rewards
    grad=-probs; grad[action]+=1; logits += .08*advantage*grad
    if step%100==0: history.append(float(probs[1]))
final=np.exp(logits-logits.max()); final/=final.sum()
assert final[1] > .9 and history[-1] > history[0]
print({"probability_better_action":round(float(final[1]),4),"learning_curve":np.round(history,3).tolist()})


## Problem 2

**Problem**: Compare $\epsilon=0.1,0.2,0.3$ in PPO and analyze the effect.

### Solution

For $r=\pi_\theta/\pi_{old}$, the clip interval is $[1-\epsilon,1+\epsilon]$. Smaller $\epsilon$ makes updates conservative; larger values permit larger changes and possible instability. The code compares exact objective values for fixed $r,A$.


In [ ]:
import numpy as np

ratios=np.linspace(.6,1.4,401); advantages=np.ones_like(ratios); results={}
for epsilon in (.1,.2,.3):
    surrogate=np.minimum(ratios*advantages,np.clip(ratios,1-epsilon,1+epsilon)*advantages)
    clipped_fraction=float(np.mean(ratios>1+epsilon)); max_objective=float(surrogate.max())
    results[epsilon]={"clipped_fraction":clipped_fraction,"max_surrogate":max_objective}
assert results[.1]["clipped_fraction"] > results[.2]["clipped_fraction"] > results[.3]["clipped_fraction"]
print({str(k):{m:round(v,4) for m,v in r.items()} for k,r in results.items()})


## Problem 3

**Problem**: Compare RM loss when chosen and rejected responses are similar versus different.

### Solution

RM loss is $-\log\sigma(r_c-r_r)$. It equals $\log2$ at zero gap, approaches zero when chosen is far better, and grows when ordering is wrong. Thus only score differences, not absolute scores, are identified.


In [ ]:
import numpy as np

gaps=np.array([0.0,0.5,2.0,-0.5]); losses=np.logaddexp(0,-gaps)
assert losses[2] < losses[1] < losses[0] < losses[3]
print([{"chosen_minus_rejected":float(g),"preference_loss":round(float(l),5)} for g,l in zip(gaps,losses)])


## Problem 4

**Problem**: Simulate policy changes for KL penalties $\beta=0,0.1,1.0$.

### Solution

In $R_{total}=R_{RM}-\beta D_{KL}$, larger $\beta$ suppresses gains from moving away from the reference. A one-dimensional concave proxy shows the optimal change scales as $1/\beta$; with $\beta=0$ there is no finite constrained optimum.


In [ ]:
import numpy as np

# Optimize reward*delta - beta*delta^2/2 on a bounded grid; delta proxies policy movement/KL.
delta=np.linspace(0,3,3001); results={}
for beta in (0.0,.1,1.0):
    objective=delta-beta*delta**2/2
    optimum=float(delta[np.argmax(objective)])
    results[beta]={"optimal_policy_shift":optimum,"kl_proxy":optimum**2/2}
assert results[1.0]["optimal_policy_shift"] < results[.1]["optimal_policy_shift"] <= results[0.0]["optimal_policy_shift"]
print({str(k):{m:round(v,3) for m,v in r.items()} for k,r in results.items()})


## Problem 5

**Problem**: Explain DPO’s advantages over RLHF from a memory perspective.

### Solution

PPO-style RLHF requires policy, reference, reward, and value models plus rollout activations. DPO usually needs only log probabilities from a trainable policy and fixed reference, eliminating reward/value models and online rollouts; model-state and activation memory are reduced.


In [ ]:
# Count simultaneously resident trainable/frozen model-sized states under explicit assumptions.
ppo={"policy":1,"reference":1,"reward":1,"value":1,"rollout_buffer":.25}
dpo={"policy":1,"reference":1,"preference_batch":.05}
ppo_total=sum(ppo.values()); dpo_total=sum(dpo.values())
assert dpo_total < ppo_total and "reward" not in dpo and "value" not in dpo
print({"units_of_model_memory":{"PPO":ppo_total,"DPO":dpo_total},"relative_reduction":round(1-dpo_total/ppo_total,3),
       "assumption":"same precision; optimizer/activation memory excluded"})
